# ML Assignment 2 — Breast Cancer Diagnosis Classification
**M.Tech (AIML) · Machine Learning · BITS Pilani WILP**

This notebook trains and evaluates five classifiers on the **Breast Cancer Wisconsin (Diagnostic)** dataset (UCI), then saves the trained models for the Streamlit app.

Workflow:
1. Load and explore the dataset
2. Train/test split (the test split is exported as `test_data.csv` for the app)
3. Train Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes and SVM
4. Evaluate each with Accuracy, AUC, Precision, Recall, F1 and MCC
5. Save the models to `model/`

In [ ]:
import os, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix,
                             classification_report, RocCurveDisplay)

RANDOM_STATE = 42
os.makedirs("model", exist_ok=True)

# Pin these exact versions in requirements.txt so the saved models load on Streamlit Cloud
print("Python      :", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)

## 1. Load the dataset

The Breast Cancer Wisconsin (Diagnostic) dataset comes from the UCI Machine Learning Repository
(https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) and is bundled with scikit-learn.

Each row describes cell nuclei measured from a digitised image of a fine needle aspirate (FNA) of a breast mass.
Ten nucleus properties (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points,
symmetry, fractal dimension) are each reported as mean, standard error and "worst" (largest) value → **30 numeric features**.

scikit-learn encodes `0 = malignant, 1 = benign`. We re-encode the target as **`diagnosis`: 1 = malignant, 0 = benign**,
so that the positive class is the disease (Precision/Recall then describe cancer detection).

In [ ]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()
df["diagnosis"] = (df["target"] == 0).astype(int)   # 1 = malignant, 0 = benign
df = df.drop(columns="target")
df.columns = [c.replace(" ", "_") for c in df.columns]

df.to_csv("breast_cancer_full.csv", index=False)
print("Shape:", df.shape)
df.head()

## 2. Exploratory data analysis

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print()
print(df["diagnosis"].value_counts().rename({0: "benign (0)", 1: "malignant (1)"}))
df.describe().T.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
df["diagnosis"].map({0: "Benign", 1: "Malignant"}).value_counts().plot.bar(
    ax=axes[0], color=["#6C8EBF", "#C2185B"], rot=0)
axes[0].set_title("Class distribution")
axes[0].set_ylabel("Count")

for label, color in [(0, "#6C8EBF"), (1, "#C2185B")]:
    sns.kdeplot(df.loc[df["diagnosis"] == label, "worst_radius"], ax=axes[1],
                fill=True, color=color, label="Malignant" if label else "Benign")
axes[1].set_title("worst_radius by diagnosis")
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": .7})
plt.title("Feature correlation matrix")
plt.tight_layout(); plt.show()

print("Top features correlated with diagnosis:")
print(df.corr()["diagnosis"].drop("diagnosis").abs().sort_values(ascending=False).head(8).round(3))

**EDA notes**
- No missing values, so no imputation is needed.
- The classes are moderately imbalanced (~63% benign / 37% malignant), so we use a **stratified** split and report MCC/F1 in addition to accuracy.
- Features have very different scales (e.g. `area` in the hundreds vs `smoothness` ≈ 0.1), so distance- and margin-based models (KNN, SVM, Logistic Regression) need standardisation.
- Size-related features (radius, perimeter, area) are highly correlated with each other — this violates the independence assumption of Naive Bayes.

## 3. Train/test split

80/20 stratified split. The 20% test portion (114 rows, features + `diagnosis`) is saved as `test_data.csv`; this is the file uploaded to the Streamlit app.

In [ ]:
FEATURES = [c for c in df.columns if c != "diagnosis"]
TARGET = "diagnosis"

X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

test_df = X_test.copy()
test_df[TARGET] = y_test
test_df.to_csv("test_data.csv", index=False)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Malignant share — train: %.3f, test: %.3f" % (y_train.mean(), y_test.mean()))

## 4. Define the models

Each model is wrapped in a `Pipeline`. Where scaling is needed, the `StandardScaler` is fitted **only on training data** and saved inside the pipeline, so the app can feed raw test features straight into the model with no separate preprocessing artifact and no data leakage.

| Model | Scaling | Key settings |
|---|---|---|
| Logistic Regression | Yes | `max_iter=1000` |
| Decision Tree | No (tree splits are scale-invariant) | `max_depth=5` to limit overfitting |
| KNN | Yes | `k=7` |
| Gaussian Naive Bayes | No | defaults |
| SVM | Yes | RBF kernel, `probability=True` for AUC |

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=7))]),
    "Naive Bayes": Pipeline([
        ("clf", GaussianNB())]),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))]),
}

## 5. Train and evaluate

In [ ]:
def evaluate(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        "Accuracy":  accuracy_score(y, y_pred),
        "AUC":       roc_auc_score(y, y_prob),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall":    recall_score(y, y_pred, zero_division=0),
        "F1":        f1_score(y, y_pred, zero_division=0),
        "MCC":       matthews_corrcoef(y, y_pred),
    }, y_pred

results, predictions = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name], predictions[name] = evaluate(model, X_test, y_test)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    print(f"{name:<24} train acc = {train_acc:.4f}   test acc = {results[name]['Accuracy']:.4f}")

results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = "ML Model Name"
results_df

### Comparison table (test set)

In [ ]:
display_df = results_df.style.highlight_max(axis=0, color="#F8BBD0").format("{:.4f}")
display_df

In [ ]:
results_df.plot.bar(figsize=(13, 5), rot=0, width=0.8, colormap="viridis")
plt.ylim(0.75, 1.01)
plt.title("Evaluation metrics by model (test set)")
plt.ylabel("Score")
plt.legend(loc="lower right", ncol=6)
plt.tight_layout(); plt.show()

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="RdPu", cbar=False, ax=ax,
                xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

In [ ]:
for name, y_pred in predictions.items():
    print("=" * 60)
    print(name)
    print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"], digits=4))

### ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("ROC curves (test set)")
plt.tight_layout(); plt.show()

### Stability check: 5-fold cross-validation on the training set

A single 114-row test set is small; one misclassified tumour moves accuracy by ~0.9%. Cross-validation on the training data shows whether the ranking is stable.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
    cv_rows.append({"Model": name, "CV F1 mean": scores.mean(), "CV F1 std": scores.std()})
pd.DataFrame(cv_rows).set_index("Model").round(4)

## 6. Save models and metrics

In [ ]:
FILE_NAMES = {
    "Logistic Regression":    "logistic_regression.pkl",
    "Decision Tree":          "decision_tree.pkl",
    "KNN":                    "knn.pkl",
    "Naive Bayes":            "naive_bayes.pkl",
    "Support Vector Machine": "svm.pkl",
}
for name, model in models.items():
    joblib.dump(model, os.path.join("model", FILE_NAMES[name]))

# Metadata the app uses to validate uploaded CSVs
meta = {"features": FEATURES, "target": TARGET,
        "class_names": {"0": "Benign", "1": "Malignant"},
        "model_files": FILE_NAMES,
        "sklearn_version": sklearn.__version__}
with open("model/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

results_df.to_csv("model/test_metrics.csv")
print(sorted(os.listdir("model")))

In [ ]:
# Sanity check: reload a saved model and confirm it reproduces the test metrics
reloaded = joblib.load("model/svm.pkl")
test_back = pd.read_csv("test_data.csv")
print(evaluate(reloaded, test_back[FEATURES], test_back[TARGET])[0])

## 7. Observations

| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | See README — fill from your own run |
| Decision Tree | |
| KNN | |
| Naive Bayes | |
| Support Vector Machine | |

*(Full observations are written in README.md.)*